# Transformacion y escalado - NHANES (Fase 2)
Cuaderno de evidencia: derivacion de etiqueta (sin filas desconocidas), one-hot, StandardScaler y PCA.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
from servicios.transformacion import DerivadorEtiqueta, SelectorVariables, CodificadorOneHot, Escalador, AnalizadorPCA, ConstructorMatriz
from utilidades.constantes import COLUMNAS_CATEGORICAS
df = pd.read_csv(Path().resolve().parent / 'outputs/cleaned/nhanes_master_cleaned.csv')
df = DerivadorEtiqueta().derivar(df)
df = df.loc[df['Disease_Indicator'].notna()].reset_index(drop=True)
df['Disease_Indicator'] = df['Disease_Indicator'].astype(int)
numPred = ['RIDAGEYR','BMXWT','BMXHT','BMXWAIST','DR1TKCAL','DR1TSUGR','DR1TPROT','DR1TCARB','DR1TFIBE','DR1TTFAT']
numCols, catCols = SelectorVariables(numPred, COLUMNAS_CATEGORICAS).seleccionar(df)
dummies = CodificadorOneHot(catCols).codificar(df)
Xnum = Escalador().ajustarTransformar(df, numCols)
pca = AnalizadorPCA()
acum, p80, p90 = pca.analizar(Xnum)
pca.graficar(Path().resolve().parent / 'outputs/figuras/pca_scree_plot.png')
y = df['Disease_Indicator']
Xfinal = ConstructorMatriz().construir(Xnum, dummies, y)
print('X:', Xfinal.shape, 'y:', y.shape)
